In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.metrics import classification_report, confusion_matrix

# 1. Wczytanie nocnych łupów
# Upewnij się, że nazwy plików zgadzają się z tymi wygenerowanymi przez skrypt
X_train = np.load("TRAIN_LANDSCAPES_X.npy")
y_train = np.load("TRAIN_LANDSCAPES_y.npy")

print(f"Dane wczytane pomyślnie!")
print(f"Kształt macierzy cech X: {X_train.shape} (Powinno być ok. 14551 x 200)")
print(f"Kształt etykiet y: {y_train.shape}")
print("-" * 50)

# 2. Walidacja krzyżowa - ostateczny test skuteczności wektoryzacji pejzażami
print("Trenowanie modelu Random Forest na Pejzażach Persystentnych...")
clf = RandomForestClassifier(n_estimators=300, max_depth=20, random_state=42, n_jobs=-1)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scores = cross_val_score(clf, X_train, y_train, cv=cv, scoring='accuracy')
print(f"ŚREDNIA DOKŁADNOŚĆ (Accuracy): {scores.mean():.4f} (+/- {scores.std() * 2:.4f})")
print("-" * 50)

# 3. Podział na zbiór testowy do głębszej analizy (Classification Report & Confusion Matrix)
classes = ['Normal', 'Pneumonia-Bacterial', 'Pneumonia-Viral', 'COVID-19', 'Tuberculosis', 'Emphysema']

X_t, X_v, y_t, y_v = train_test_split(X_train, y_train, test_size=0.2, random_state=42, stratify=y_train)

clf.fit(X_t, y_t)
preds = clf.predict(X_v)

print("Raport klasyfikacji (Precision, Recall, F1-Score):")
print(classification_report(y_v, preds, target_names=classes))

# 4. Rysowanie Macierzy Pomyłek
cm = confusion_matrix(y_v, preds)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=classes, yticklabels=classes,
            annot_kws={"size": 12})
plt.title('Macierz Pomyłek: Klasyfikacja chorób płuc za pomocą Pejzaży Persystentnych', fontsize=14)
plt.ylabel('Rzeczywista klasa (Ground Truth)', fontsize=12)
plt.xlabel('Przewidywana klasa (Prediction)', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()